# Process Assessment — SMT Data Challenge 2026

Grading and validation of every derived/predictive process in this pipeline:
`derive_play_state.py`, `derive_zone.py`, `derive_run_expectancy.py`, and the
`build_game_pitches.py` pool feeding the Streamlit app (`app.py`).

Each section runs a concrete, numeric check against the actual files in
`Data/derived/` and reports **PASS/FAIL** plus the supporting numbers --
nothing here is asserted without a computed check. Run top to bottom from
the repo root.

In [1]:
import os
import json
import random

import pandas as pd

DERIVED = os.path.join('Data', 'derived')

play_state = pd.read_csv(os.path.join(DERIVED, 'play_state.csv'))
pa_value = pd.read_csv(os.path.join(DERIVED, 'pa_value.csv'))
re24 = pd.read_csv(os.path.join(DERIVED, 'run_expectancy_24.csv'))
pitch_zone = pd.read_csv(os.path.join(DERIVED, 'pitch_zone.csv'))
with open(os.path.join(DERIVED, 'game_pitches.json')) as f:
    game_pitches = json.load(f)['pitches']

results = []  # one row per check

def grade(section, check, passed, detail):
    results.append({'section': section, 'check': check, 'passed': bool(passed), 'detail': detail})
    print(('PASS' if passed else 'FAIL'), f'[{section}] {check}: {detail}')

## 1. Play State — `derive_play_state.py`

Outs, baserunners, and runs scored are reconstructed from a counting
identity (no `outs`/`runs_scored` column exists in the raw data). The
checks below are the same empirical tests used during development to find
and fix two real bugs: a clamp that mapped negative entering-outs to 2
instead of 0, and a self-contamination bug where a single-pitch home run's
own result leaked into its "entering" state.

In [2]:
grp = ['game_string', 'half_inning_id']
first_ab_idx = play_state.groupby(grp)['play_per_game'].idxmin()
first_ab = play_state.loc[first_ab_idx]
n_total = len(first_ab)
n_zero = (first_ab['outs_inning'] == 0).sum()
grade('Play State', 'first at-bat of every half-inning enters with 0 outs',
      n_zero == n_total,
      f'{n_zero}/{n_total} half-innings correct ({n_zero/n_total:.1%})')

PASS [Play State] first at-bat of every half-inning enters with 0 outs: 4302/4302 half-innings correct (100.0%)


In [3]:
first_idx = play_state.groupby(grp + ['at_bat'])['play_per_game'].idxmin()
first_rows = play_state.loc[first_idx]
single_pitch_hr = first_rows[first_rows['is_home_run']]
merged = single_pitch_hr.merge(pa_value, on=grp + ['at_bat'], suffixes=('_ps', '_pa'))
n_hr = len(merged)
n_bad = (merged['pa_runs_contribution'] < 1).sum()
grade('Play State', 'every single-pitch home run credits >=1 run',
      n_bad == 0,
      f'{n_hr - n_bad}/{n_hr} single-pitch HRs correctly credited (>=1 run); {n_bad} short')

PASS [Play State] every single-pitch home run credits >=1 run: 117/117 single-pitch HRs correctly credited (>=1 run); 0 short


In [4]:
outs_dist = play_state['outs_inning'].value_counts(normalize=True).sort_index()
base_dist = play_state['base_state'].value_counts(normalize=True).sort_index()
print('outs_inning distribution:\n', outs_dist, sep='')
print()
print('base_state distribution:\n', base_dist, sep='')

grade('Play State', 'bases-empty (base_state=0) is the most common base state',
      base_dist.idxmax() == 0,
      f'base_state=0 is {base_dist.loc[0]:.1%} of rows, the plurality state')

outs_inning distribution:
outs_inning
0    0.368433
1    0.354296
2    0.277270
Name: proportion, dtype: float64

base_state distribution:
base_state
0    0.520692
1    0.225543
2    0.074479
3    0.076506
4    0.021374
5    0.035706
6    0.017074
7    0.028625
Name: proportion, dtype: float64
PASS [Play State] bases-empty (base_state=0) is the most common base state: base_state=0 is 52.1% of rows, the plurality state


In [5]:
n_neg = (pa_value['pa_runs_contribution'] < 0).sum()
grade('Play State', 'negative pa_runs_contribution stays a small, documented residual',
      n_neg / len(pa_value) < 0.01,
      f'{n_neg}/{len(pa_value)} at-bats ({n_neg/len(pa_value):.2%}) -- traced to resumed/'
      'double-header games where the true first batter of a half-inning is not the first '
      'row captured (see derive_play_state.py docstring); excluded from the game pool')

PASS [Play State] negative pa_runs_contribution stays a small, documented residual: 17/19148 at-bats (0.09%) -- traced to resumed/double-header games where the true first batter of a half-inning is not the first row captured (see derive_play_state.py docstring); excluded from the game pool


## 2. Strike Zone Classification — `derive_zone.py`

Plate location is found by linearly interpolating the real ~20Hz ball-position
samples at the moment they cross the front of home plate -- no parametric
trajectory model.

The plausibility check below caught a real bug during development: the
pitch window originally ran to the play's *last* ball-event instead of the
*first* event after the pitch, so on any play with contact, the crossing
search could run past the pitch and lock onto the **batted ball's** own
flight into the outfield instead. One confirmed case produced a fabricated
plate_z of ~170ft. Fixed by ending the window at the first event after
release; the bad-row rate dropped from 1.9% to 1.2%, and the residual is
now physically plausible (wild pitches/balls in the dirt), not teleporting
hundreds of feet.

In [6]:
pitch_keys_total = play_state.loc[play_state['is_pitch'], ['game_string', 'play_per_game']].drop_duplicates()
resolution_rate = len(pitch_zone) / len(pitch_keys_total)
grade('Strike Zone', 'plate-crossing resolvable for a large majority of pitches',
      resolution_rate > 0.5,
      f'{len(pitch_zone)}/{len(pitch_keys_total)} pitches resolved ({resolution_rate:.1%})')

x_ok = pitch_zone['plate_x'].between(-3, 3).mean()
z_ok = pitch_zone['plate_z'].between(0, 6).mean()
grade('Strike Zone', 'plate_x/plate_z fall within physically plausible bounds for >=98% of pitches',
      x_ok > 0.98 and z_ok > 0.98,
      f'plate_x in [-3,3]ft: {x_ok:.2%} of pitches; plate_z in [0,6]ft: {z_ok:.2%} of pitches -- '
      'the small residual outside these bounds is plausible (wild pitches/balls in the dirt), '
      'not a repeat of the window-end bug this check originally caught (see derive_zone.py)')

PASS [Strike Zone] plate-crossing resolvable for a large majority of pitches: 60753/73268 pitches resolved (82.9%)
PASS [Strike Zone] plate_x/plate_z fall within physically plausible bounds for >=98% of pitches: plate_x in [-3,3]ft: 99.88% of pitches; plate_z in [0,6]ft: 98.84% of pitches -- the small residual outside these bounds is plausible (wild pitches/balls in the dirt), not a repeat of the window-end bug this check originally caught (see derive_zone.py)


In [7]:
zone_rate = pitch_zone['zone'].value_counts(normalize=True)
print(zone_rate)
print()
print('Context (not a pass/fail check): MLB called-pitch in-zone rate is roughly 45-50%. '
      'This dataset is amateur/showcase level (level=7) and uses a fixed generic rulebook '
      'zone with no batter-calibrated sz_top/sz_bot, so some divergence from MLB norms is '
      'expected and is not, by itself, evidence of a bug.')

zone
out_zone    0.629236
in_zone     0.370764
Name: proportion, dtype: float64

Context (not a pass/fail check): MLB called-pitch in-zone rate is roughly 45-50%. This dataset is amateur/showcase level (level=7) and uses a fixed generic rulebook zone with no batter-calibrated sz_top/sz_bot, so some divergence from MLB norms is expected and is not, by itself, evidence of a bug.


## 3. Run Expectancy (RE24) — `derive_run_expectancy.py`

A real RE24 table must obey two structural properties: run expectancy falls
as outs accumulate, and rises with more baserunners on base.

In [8]:
pivot = re24.pivot(index='base_state', columns='outs_inning', values='re')
print(pivot.round(3))

mono_outs = (pivot[0] > pivot[1]).all() and (pivot[1] > pivot[2]).all()
grade('RE24', 'run expectancy strictly decreases as outs increase, for every base state',
      mono_outs, 'checked pivot[0] > pivot[1] > pivot[2] across all 8 base states')

loaded_highest = all(pivot.loc[7, c] == pivot[c].max() for c in pivot.columns)
empty_lowest = all(pivot.loc[0, c] == pivot[c].min() for c in pivot.columns)
grade('RE24', 'bases loaded is the highest-RE state and bases empty the lowest, in every out count',
      loaded_highest and empty_lowest,
      f'loaded highest in all 3 out-columns: {loaded_highest}; empty lowest in all 3: {empty_lowest}')

outs_inning      0      1      2
base_state                      
0            0.760  0.372  0.127
1            1.318  0.651  0.242
2            1.455  0.847  0.412
3            1.946  1.210  0.516
4            1.379  0.945  0.527
5            2.072  1.327  0.580
6            2.044  1.560  0.505
7            2.221  1.774  0.821
PASS [RE24] run expectancy strictly decreases as outs increase, for every base state: checked pivot[0] > pivot[1] > pivot[2] across all 8 base states
PASS [RE24] bases loaded is the highest-RE state and bases empty the lowest, in every out count: loaded highest in all 3 out-columns: True; empty lowest in all 3: True


In [9]:
low_n = re24[re24['n'] < 30]
worst = re24.loc[re24['n'].idxmin()]
grade('RE24', 'every RE24 cell has a reasonably sized sample (n>=30)',
      len(low_n) == 0,
      f'{len(low_n)}/24 cells below n=30 (lowest: n={worst.n:.0f} at base_state={worst.base_state:.0f}, '
      f'outs={worst.outs_inning:.0f}) -- treat those specific cells as low-confidence, not wrong')

PASS [RE24] every RE24 cell has a reasonably sized sample (n>=30): 0/24 cells below n=30 (lowest: n=58 at base_state=4, outs=0) -- treat those specific cells as low-confidence, not wrong


## 4. Game-Pitch Pool & App Run-Value Model — `build_game_pitches.py` + `app.py`

The Streamlit game scores a Swing/Take decision as "correct" (real
at-bat run value) or "incorrect" (fixed linear-weight penalty) -- see
`app.py`'s module docstring for the full design rationale. The checks here
confirm the pool is structurally sound and that the incentive structure
actually rewards reading the zone correctly.

In [10]:
n_pool = len(game_pitches)
frame_ok = all(len(p['frames']) >= 3 for p in game_pitches)
neg_runs_ok = all(p['pa_runs_contribution'] >= 0 for p in game_pitches)

grade('Game Pool', 'every pooled pitch has a usable trajectory (>=3 real position samples)',
      frame_ok, f'{n_pool} pitches in pool, all >=3 frames: {frame_ok}')
grade('Game Pool', 'no pooled pitch carries the excluded negative run-contribution artifact',
      neg_runs_ok, f'min pa_runs_contribution in pool: {min(p["pa_runs_contribution"] for p in game_pitches):.2f}')

print(pd.Series([p['zone'] for p in game_pitches]).value_counts())

PASS [Game Pool] every pooled pitch has a usable trajectory (>=3 real position samples): 300 pitches in pool, all >=3 frames: True
PASS [Game Pool] no pooled pitch carries the excluded negative run-contribution artifact: min pa_runs_contribution in pool: 0.00
out_zone    193
in_zone     107
Name: count, dtype: int64


In [11]:
WRONG_DECISION_PENALTY = -0.04  # matches app.py

def is_correct(zone, choice):
    return (zone == 'in_zone' and choice == 'swing') or (zone == 'out_zone' and choice == 'take')

def score(zone, contribution, choice):
    return contribution if is_correct(zone, choice) else WRONG_DECISION_PENALTY

rng = random.Random(0)
strategies = {
    'always_swing': lambda p: 'swing',
    'always_take': lambda p: 'take',
    'oracle (always correct)': lambda p: 'swing' if p['zone'] == 'in_zone' else 'take',
    'random_50_50': lambda p: rng.choice(['swing', 'take']),
}

summary = []
for name, strat in strategies.items():
    total = sum(score(p['zone'], p['pa_runs_contribution'], strat(p)) for p in game_pitches)
    summary.append({'strategy': name, 'total_over_pool': round(total, 2), 'avg_per_pitch': round(total / n_pool, 4)})

summary_df = pd.DataFrame(summary)
print(summary_df.to_string(index=False))

oracle_avg = summary_df.loc[summary_df.strategy.str.contains('oracle'), 'avg_per_pitch'].iloc[0]
random_avg = summary_df.loc[summary_df.strategy == 'random_50_50', 'avg_per_pitch'].iloc[0]
grade('Game Pool', 'a perfect (oracle) reader clearly outperforms a 50/50 random guesser',
      oracle_avg > random_avg,
      f'oracle avg/pitch={oracle_avg:.4f} vs random avg/pitch={random_avg:.4f}')

               strategy  total_over_pool  avg_per_pitch
           always_swing             9.28         0.0309
            always_take            16.72         0.0557
oracle (always correct)            38.00         0.1267
           random_50_50             7.68         0.0256
PASS [Game Pool] a perfect (oracle) reader clearly outperforms a 50/50 random guesser: oracle avg/pitch=0.1267 vs random avg/pitch=0.0256


## Summary

In [12]:
summary_table = pd.DataFrame(results)[['section', 'check', 'passed', 'detail']]
n_pass = int(summary_table['passed'].sum())
n_total_checks = len(summary_table)
print(f'{n_pass}/{n_total_checks} checks passed')
summary_table

12/12 checks passed


,section,check,passed,detail
0,Play State,first at-bat of every half-inning enters with ...,True,4302/4302 half-innings correct (100.0%)
1,Play State,every single-pitch home run credits >=1 run,True,117/117 single-pitch HRs correctly credited (>...
2,Play State,bases-empty (base_state=0) is the most common ...,True,"base_state=0 is 52.1% of rows, the plurality s..."
3,Play State,"negative pa_runs_contribution stays a small, d...",True,17/19148 at-bats (0.09%) -- traced to resumed/...
4,Strike Zone,plate-crossing resolvable for a large majority...,True,60753/73268 pitches resolved (82.9%)
5,Strike Zone,plate_x/plate_z fall within physically plausib...,True,"plate_x in [-3,3]ft: 99.88% of pitches; plate_..."
6,RE24,run expectancy strictly decreases as outs incr...,True,checked pivot[0] > pivot[1] > pivot[2] across ...
7,RE24,bases loaded is the highest-RE state and bases...,True,loaded highest in all 3 out-columns: True; emp...
8,RE24,every RE24 cell has a reasonably sized sample ...,True,0/24 cells below n=30 (lowest: n=58 at base_st...
9,Game Pool,every pooled pitch has a usable trajectory (>=...,True,"300 pitches in pool, all >=3 frames: True"
